# FocusRoom — 03: Model Evaluation
**Goal:** Deep-dive into model performance on the held-out test set.

This is the notebook you show to your graders and hackathon judges. It produces:
- Per-class precision / recall / F1
- Confusion matrix heatmap
- Grad-CAM visualisations (what part of the eye does the CNN actually look at?)
- Inference speed benchmark
- Model size summary

**Prerequisite:** `02_model_training.ipynb` must have produced `checkpoints/best_model.keras`.

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import time

CONFIG = {
    'data_dir':        '../data/processed',
    'best_checkpoint': '../checkpoints/best_model.keras',
    'img_size':        (48, 48),
    'batch_size':      32,
    'class_names':     ['focused', 'distracted', 'closed'],
    'seed':            42,
}
CLASSES = CONFIG['class_names']
COLOURS = ['#2D6A4F', '#E76F51', '#1A1A2E']

print('Libraries loaded ✓')
print(f'TensorFlow {tf.__version__}')

## 1. Load Model & Test Data

In [ ]:
model = keras.models.load_model(CONFIG['best_checkpoint'])
print(f'Model loaded from: {CONFIG["best_checkpoint"]}')
print(f'Parameters: {model.count_params():,}')

datagen  = ImageDataGenerator(rescale=1.0/255.0)
test_gen = datagen.flow_from_directory(
    os.path.join(CONFIG['data_dir'], 'test'),
    target_size=CONFIG['img_size'],
    color_mode='grayscale',
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False,
    seed=CONFIG['seed'],
)
print(f'Test samples: {test_gen.samples:,}')

## 2. Full Prediction Run

In [ ]:
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_gen.classes
y_conf       = np.max(y_pred_probs, axis=1)

acc = (y_pred == y_true).mean()
print(f'\nTest accuracy : {acc*100:.2f}%')
print(f'Target (≥88%) : {"✓ MET" if acc >= 0.88 else "✗ NOT MET"}')

## 3. Per-Class Report

In [ ]:
print('Classification Report:')
print('─'*55)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f'Macro F1 score : {macro_f1:.4f}')

## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Eye State CNN — Confusion Matrix (Test Set)', fontsize=13, fontweight='bold')

for ax, data, title, fmt in [
    (axes[0], cm,      'Raw Counts',       'd'),
    (axes[1], cm_norm, 'Row-Normalised %', '.2%'),
]:
    im = ax.imshow(data, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=20, ha='right')
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    thresh = data.max() / 2.0
    for r in range(len(CLASSES)):
        for c in range(len(CLASSES)):
            val = data[r, c]
            txt = format(val, fmt) if fmt == '.2%' else str(val)
            ax.text(c, r, txt, ha='center', va='center', fontsize=11,
                    color='white' if val > thresh else 'black', fontweight='bold')

plt.tight_layout()
plt.savefig('../checkpoints/confusion_matrix_notebook.png', dpi=150)
plt.show()
print('Saved → checkpoints/confusion_matrix_notebook.png')

## 5. Grad-CAM — What Does the Model Actually Look At?
Grad-CAM creates a heatmap showing which pixels in the eye crop were most important for the model's decision. This is your #1 explainability tool for graders — it proves the model is learning eye features, not background artefacts.

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name='conv3b', pred_index=None):
    """
    Compute Grad-CAM heatmap for a single image.
    img_array: numpy array, shape (1, 48, 48, 1), values in [0, 1]
    """
    # Build a model that outputs both the last conv features AND the final prediction
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradient of the predicted class w.r.t. the last conv feature map
    grads = tape.gradient(class_channel, conv_outputs)

    # Pool gradients over spatial dimensions → importance weight per filter
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight the conv outputs by the pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)  # ReLU + normalise
    return heatmap.numpy()


def overlay_heatmap(orig_img, heatmap, alpha=0.5):
    """
    Overlay a Grad-CAM heatmap onto the original grayscale image.
    Returns an RGB image with a coloured overlay.
    """
    # Resize heatmap to match image size
    heatmap_resized = np.array(
        tf.image.resize(heatmap[..., np.newaxis], [orig_img.shape[0], orig_img.shape[1]])
    ).squeeze()

    # Apply jet colormap to heatmap
    heatmap_col = cm.jet(heatmap_resized)[:, :, :3]  # drop alpha channel

    # Convert grayscale original to RGB for overlay
    orig_rgb = np.stack([orig_img] * 3, axis=-1)  # (H, W, 3)

    # Blend
    overlay = (1 - alpha) * orig_rgb + alpha * heatmap_col
    return np.clip(overlay, 0, 1)


print('Grad-CAM functions defined ✓')
print('Note: uses layer named "conv3b" — update if you renamed layers in your model.')

In [ ]:
# Generate Grad-CAM for 2 correct examples per class
# Collect one correctly-classified image per class
from PIL import Image as PILImage

n_per_class = 2
fig, axes = plt.subplots(len(CLASSES), n_per_class * 2, figsize=(14, 7))
fig.suptitle('Grad-CAM — What the CNN Looks At (Original | Attention Heatmap)', fontsize=12, fontweight='bold')

for cls_idx, cls_name in enumerate(CLASSES):
    # Find indices where model is correct AND this class
    correct_mask = (y_pred == y_true) & (y_true == cls_idx)
    correct_indices = np.where(correct_mask)[0]

    if len(correct_indices) == 0:
        print(f'  No correct predictions found for class: {cls_name}')
        continue

    chosen = correct_indices[:n_per_class]

    for col_pair, img_idx in enumerate(chosen):
        fpath = test_gen.filepaths[img_idx]
        orig_pil = PILImage.open(fpath).convert('L').resize((48, 48))
        orig_arr = np.array(orig_pil) / 255.0  # (48, 48), values in [0,1]
        img_batch = orig_arr[np.newaxis, :, :, np.newaxis].astype(np.float32)

        # Grad-CAM
        try:
            heatmap = get_gradcam_heatmap(model, img_batch, pred_index=cls_idx)
            overlay = overlay_heatmap(orig_arr, heatmap, alpha=0.45)

            col_orig = col_pair * 2
            col_heat = col_pair * 2 + 1

            # Original
            axes[cls_idx][col_orig].imshow(orig_arr, cmap='gray', vmin=0, vmax=1)
            axes[cls_idx][col_orig].set_title(f'{cls_name}\nconf={y_conf[img_idx]:.2f}', fontsize=8)
            axes[cls_idx][col_orig].axis('off')

            # Heatmap overlay
            axes[cls_idx][col_heat].imshow(overlay)
            axes[cls_idx][col_heat].set_title('Grad-CAM', fontsize=8)
            axes[cls_idx][col_heat].axis('off')

        except Exception as e:
            print(f'  Grad-CAM failed for {cls_name} idx {img_idx}: {e}')
            print('  Check that the last conv layer is named "conv3b" in your model.')

plt.tight_layout()
plt.savefig('../checkpoints/gradcam.png', dpi=150)
plt.show()
print('Saved → checkpoints/gradcam.png')
print('\nWhat to look for:')
print('  ✓ Red/yellow areas should be on the iris, eyelid, or gaze region')
print('  ✗ If hotspots are on background/corners — model is cheating, dataset has artefacts')

## 6. Inference Speed Benchmark

In [ ]:
# Simulate real FocusRoom usage: single image at a time
N_BENCH = 200
test_gen.reset()
bench_images = []
for batch_x, _ in test_gen:
    for img in batch_x:
        bench_images.append(img)
        if len(bench_images) >= N_BENCH:
            break
    if len(bench_images) >= N_BENCH:
        break

latencies = []
for img in bench_images:
    inp = np.expand_dims(img, axis=0)  # (1, 48, 48, 1)
    t0 = time.perf_counter()
    _ = model.predict(inp, verbose=0)
    latencies.append((time.perf_counter() - t0) * 1000)

avg = np.mean(latencies)
p95 = np.percentile(latencies, 95)
print(f'Inference speed over {N_BENCH} single images:')
print(f'  Average : {avg:.2f} ms')
print(f'  P95     : {p95:.2f} ms')
print(f'  SLA     : < 150ms')
print(f'  Status  : {"✓ PASS" if avg < 150 else "✗ FAIL — use TFLite for speedup"}')

# Latency distribution plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=40, color='#1A1A2E', alpha=0.8, edgecolor='white')
ax.axvline(avg, color='#E63946', linestyle='--', linewidth=2, label=f'Mean: {avg:.1f}ms')
ax.axvline(150, color='#E76F51', linestyle=':', linewidth=2, label='SLA: 150ms')
ax.set_title('Inference Latency Distribution', fontweight='bold')
ax.set_xlabel('Milliseconds per image'); ax.set_ylabel('Count')
ax.legend(); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../checkpoints/latency_distribution.png', dpi=120)
plt.show()

## 7. Model Size Summary

In [ ]:
import os

def dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

print('Model size summary:')
print('─'*45)

paths = {
    'Keras (.keras)':          '../checkpoints/best_model.keras',
    'TFLite float32':          '../exports/tflite/eye_cnn_float32.tflite',
    'TFLite int8 (quantised)': '../exports/tflite/eye_cnn_int8.tflite',
    'TF.js (total)':           '../exports/tfjs_model/',
}

for label, path in paths.items():
    if os.path.isfile(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f'  {label:28s}: {size:.2f} MB')
    elif os.path.isdir(path):
        size = dir_size_mb(path)
        print(f'  {label:28s}: {size:.2f} MB')
    else:
        print(f'  {label:28s}: not yet exported (run export.py)')

print(f'\n  Total parameters: {model.count_params():,}')
print(f'  Input shape     : {model.input_shape}')
print(f'  Output shape    : {model.output_shape}')

## 8. Final Summary (Hackathon/Grader Summary)

In [ ]:
report_dict = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)

print('╔══════════════════════════════════════════════╗')
print('║   FocusRoom — Eye State CNN Evaluation Report ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  Test Accuracy   : {acc*100:.2f}%  (target ≥ 88%)      ║')
print(f'║  Macro F1 Score  : {macro_f1:.4f}                        ║')
print(f'║  Inference (avg) : {avg:.1f}ms  (SLA < 150ms)    ║')
print('╠══════════════════════════════════════════════╣')
print('║  Per-Class F1 Scores:                        ║')
for cls in CLASSES:
    f1 = report_dict[cls]['f1-score']
    print(f'║    {cls:12s}: {f1:.4f}                         ║')
print('╠══════════════════════════════════════════════╣')
print('║  Saved outputs:                              ║')
print('║    checkpoints/confusion_matrix_notebook.png ║')
print('║    checkpoints/gradcam.png                   ║')
print('║    checkpoints/latency_distribution.png      ║')
print('╠══════════════════════════════════════════════╣')
print('║  Next → python ../export.py                  ║')
print('╚══════════════════════════════════════════════╝')